# Seminar 2 — Categorical Encoding Lab

**Цель:** увидеть, что encoding — это не техническая мелочь, а часть model specification.

### Must know
- one-hot encoding;
- ordinal encoding;
- unseen categories;
- почему произвольная нумерация nominal categories опасна.

### Should know
- frequency encoding;
- target encoding;
- leakage.

### Advanced
- out-of-fold target encoding;
- smoothing;
- high-cardinality features.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

df = pd.read_csv("data/categorical_demo.csv")
df.head()

,experience,industry,city,education,salary
0,21,Banking,Moscow,Master,280.77
1,5,Retail,Moscow,Master,137.80
2,17,IT,Saint Petersburg,Bachelor,220.64
3,0,Manufacturing,Moscow,Master,162.03
4,14,IT,Moscow,Bachelor,236.82


## 1. Economic problem

Хотим прогнозировать salary по:
- experience,
- industry,
- city,
- education.

Но linear regression принимает числа.

Как передать категорию `industry`?

## 2. Наивный ordinal encoding

Пусть:

- Banking → 0
- IT → 1
- Retail → 2
- Manufacturing → 3

Проблема: модель начинает воспринимать эти коды как числа с расстояниями.

In [2]:
mapping1 = {"Banking":0, "IT":1, "Retail":2, "Manufacturing":3}
mapping2 = {"Retail":0, "Manufacturing":1, "Banking":2, "IT":3}

tmp = df.copy()
tmp["industry_code_1"] = tmp["industry"].map(mapping1)
tmp["industry_code_2"] = tmp["industry"].map(mapping2)

X1 = tmp[["experience","industry_code_1"]]
X2 = tmp[["experience","industry_code_2"]]
y = tmp["salary"]

m1 = LinearRegression().fit(X1,y)
m2 = LinearRegression().fit(X2,y)

print("Model 1 coefficients:", m1.coef_)
print("Model 2 coefficients:", m2.coef_)

Model 1 coefficients: [ 5.07253247 -9.30226708]
Model 2 coefficients: [ 5.12066697 16.26855112]


### Вопрос
Почему простая перенумерация categories не должна менять экономическую задачу, но меняет linear regression?

## 3. One-hot encoding

Для nominal categories обычно естественнее создать отдельные dummy indicators.

In [3]:
X = df[["experience","industry"]]
y = df["salary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pre = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["industry"])
], remainder="passthrough")

pipe = Pipeline([
    ("prep", pre),
    ("model", LinearRegression())
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print("Test RMSE:", mean_squared_error(y_test, pred)**0.5)

Test RMSE: 20.496397902112133


## 4. Что означает baseline category?

При `drop="first"` одна категория становится reference group.

Коэффициенты остальных dummy variables интерпретируются **относительно baseline**.

Это связывает ML encoding и econometric dummy variables.

## 5. Unseen category

Что делать, если в test появляется новая категория?

`OneHotEncoder(handle_unknown="ignore")` не падает с ошибкой, но новая категория не получает отдельного learned effect.

In [4]:
demo = pd.DataFrame({
    "experience":[5,5],
    "industry":["IT","Telecom"]
})

pipe.predict(demo)

/Users/irinakharkova/my_projects/ml_course/2 семинар линейнай регрессия/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


array([179.94659841, 158.50963022])

## 6. Frequency encoding

Категория кодируется своей частотой:

$$
x_c = \frac{n_c}{n}.
$$

Плюсы:
- одна числовая колонка;
- удобно при high cardinality.

Минусы:
- две категории с одинаковой частотой становятся неразличимыми;
- частота не обязана иметь экономический смысл.

In [ ]:
freq = df["industry"].value_counts(normalize=True)
df_freq = df.copy()
df_freq["industry_freq"] = df_freq["industry"].map(freq)

df_freq[["industry","industry_freq"]].drop_duplicates().sort_values("industry_freq")

## 7. Target encoding

$$
TE(c)=E[Y|X=c].
$$

Интуитивно очень привлекательный подход.

Но target используется для создания feature → возникает риск leakage.

In [5]:
te = df.groupby("industry")["salary"].mean()
te

industry
Banking          197.085234
IT               219.910252
Manufacturing    181.155402
Retail           172.127757
Name: salary, dtype: float64

## 8. Leakage experiment

Неправильно:
1. посчитать target mean на всей выборке;
2. потом сделать train/test split.

Правильно:
1. split;
2. считать encoding только по train;
3. применить его к validation/test.

In [6]:
train, test = train_test_split(df, test_size=0.25, random_state=42)

global_mean = train["salary"].mean()
te_train = train.groupby("industry")["salary"].mean()

train_te = train.copy()
test_te = test.copy()

train_te["industry_te"] = train_te["industry"].map(te_train).fillna(global_mean)
test_te["industry_te"] = test_te["industry"].map(te_train).fillna(global_mean)

m = LinearRegression().fit(train_te[["experience","industry_te"]], train_te["salary"])
pred = m.predict(test_te[["experience","industry_te"]])

mean_squared_error(test_te["salary"], pred)**0.5

21.00370302705704

## 9. Advanced: out-of-fold target encoding

Для training observations лучше не использовать target самого observation при построении его encoded feature.

Идея:
- fold 1 кодируется статистиками folds 2–K;
- fold 2 — статистиками остальных folds;
- и т.д.

In [7]:
def oof_target_encode(df, col, target, n_splits=5, random_state=42):
    out = pd.Series(index=df.index, dtype=float)
    global_mean = df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for train_idx, valid_idx in kf.split(df):
        tr = df.iloc[train_idx]
        va = df.iloc[valid_idx]
        means = tr.groupby(col)[target].mean()
        out.iloc[valid_idx] = va[col].map(means).fillna(global_mean)
    return out

df_oof = df.copy()
df_oof["industry_te_oof"] = oof_target_encode(df_oof, "industry", "salary")
df_oof.head()

,experience,industry,city,education,salary,industry_te_oof
0,21,Banking,Moscow,Master,280.77,198.579494
1,5,Retail,Moscow,Master,137.80,178.133816
2,17,IT,Saint Petersburg,Bachelor,220.64,224.576082
3,0,Manufacturing,Moscow,Master,162.03,183.031563
4,14,IT,Moscow,Bachelor,236.82,220.119556


## 10. Advanced: smoothing

Для редкой категории sample mean может быть очень нестабилен.

Один из вариантов:

$$
TE_c=
\frac{n_c\bar y_c + \lambda\bar y}{n_c+\lambda}.
$$

Если наблюдений мало → estimate тянется к global mean.

Если много → больше доверяем category mean.

## 11. Сравнительная таблица

| Encoding | # features | Interpretability | Unseen categories | Leakage risk |
|---|---:|---|---|---|
| Ordinal | 1 | низкая для nominal | да | низкий |
| One-hot | K-1 / K | высокая | manageable | низкий |
| Frequency | 1 | средняя | manageable | низкий |
| Target | 1 | средняя | manageable | **высокий** |

> **Главный вывод:** encoding — часть model assumptions.